In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 3 — Skewness, Outliers & Multicollinearity

**Project:** Wholesale Customer Segmentation

Phase 3 - Handling Skewness, Outliers & Multicollinearity
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 8.   Examine skewness and outliers
 8.5. Decide and apply outlier treatment
 8.6. Address multicollinearity (decide: keep, drop, or note for PCA)

Depends on: phase1_setup.py (df_raw), phase2_eda.py (SPEND_COLS, select_clustering_features)
Outputs: PNG figures saved to ./figures/, printed analysis + documented decisions

Note on dependencies: VIF is computed manually via sklearn LinearRegression
(VIF = 1 / (1 - R^2) of each feature regressed on the rest), because
`statsmodels` is not available in this environment and there is no network
access to install it. This is mathematically equivalent to
statsmodels.stats.outliers_influence.variance_inflation_factor.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression



# Recreate upstream constants/data locally so this notebook is standalone.
DATA_PATH = "data/raw/wholesale_customers.csv"
df_raw = pd.read_csv(DATA_PATH)
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
FIG_DIR = "figures"
Path = __import__("pathlib").Path
Path(FIG_DIR).mkdir(exist_ok=True)

def select_clustering_features(df):
    return df[SPEND_COLS].copy()

sns.set_style("whitegrid")

## SECTION 1: Skewness Analysis (Step 8)

In [ ]:
# SECTION 1: Skewness Analysis (Step 8)
# ===========================================================================
def analyze_skewness(df: pd.DataFrame) -> pd.Series:
    """Quantify skewness per spend column to justify transformation need."""
    print("=" * 70)
    print("SECTION 1: SKEWNESS ANALYSIS")
    print("=" * 70)

    skew_vals = df[SPEND_COLS].skew().sort_values(ascending=False)
    print("Skewness per feature (Pearson's moment coefficient):")
    print(skew_vals.round(2))

    print("\nInterpretation guide: |skew| > 1 = highly skewed; 0.5-1 = moderate; <0.5 = ~symmetric")
    for col, val in skew_vals.items():
        level = "HIGH" if abs(val) > 1 else ("MODERATE" if abs(val) > 0.5 else "LOW")
        print(f"  {col:<20} skew = {val:6.2f}  -> {level} skew")

    print("\n[FINDING] All 6 features exceed skew > 2 (Delicassen most extreme "
          f"at {skew_vals.max():.2f}). The distributions are heavily right-skewed. "
          "Because K-Means minimizes squared Euclidean distances, extreme values "
          "can disproportionately influence centroid positions. A variance-stabilizing "
          "log1p transform is therefore justified as a robustness step.")
    return skew_vals

## SECTION 2: Outlier Detection via IQR Method (Step 8)

In [ ]:
# SECTION 2: Outlier Detection via IQR Method (Step 8)
# ===========================================================================
def detect_outliers_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """Detect outliers per spend column using the 1.5*IQR rule."""
    print("\n" + "=" * 70)
    print("SECTION 2: OUTLIER DETECTION (IQR METHOD)")
    print("=" * 70)

    rows = []
    for col in SPEND_COLS:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (df[col] < lower) | (df[col] > upper)
        rows.append({
            "feature": col,
            "Q1": q1, "Q3": q3, "IQR": iqr,
            "lower_bound": lower, "upper_bound": upper,
            "n_outliers": int(mask.sum()),
            "pct_outliers": round(100 * mask.sum() / len(df), 1),
        })

    outlier_summary = pd.DataFrame(rows).set_index("feature")
    print(outlier_summary.round(1).to_string())

    total_flagged_rows = (
        (df[SPEND_COLS] < outlier_summary["lower_bound"]) |
        (df[SPEND_COLS] > outlier_summary["upper_bound"])
    ).any(axis=1).sum()
    print(f"\n[FINDING] {outlier_summary['n_outliers'].min()}-{outlier_summary['n_outliers'].max()} "
          f"outliers per column (IQR rule). {total_flagged_rows} of {len(df)} customers "
          "({:.1f}%) have at least one outlying spend value — concentrated in "
          "high-spend 'bulk buyer' accounts, not data errors.".format(
              100 * total_flagged_rows / len(df)))
    return outlier_summary

## SECTION 3: Outlier Treatment Decision (Step 8.5)

In [ ]:
# SECTION 3: Outlier Treatment Decision (Step 8.5)
# ===========================================================================
def document_outlier_decision() -> str:
    """State and justify the outlier-handling decision for this project."""
    print("\n" + "=" * 70)
    print("SECTION 3: OUTLIER TREATMENT DECISION")
    print("=" * 70)

    decision = (
        "DECISION: Retain all outlier records — do NOT delete or cap them.\n\n"
        "RATIONALE:\n"
        "  1. Outliers are concentrated in high-spend accounts, which plausibly "
        "represent genuine large/bulk-buying customers rather than data-entry "
        "errors. Removing them would delete exactly the customer behavior most "
        "relevant to segmentation (e.g. distinguishing large Horeca accounts "
        "from small retail accounts).\n"
        "  2. No documented evidence (data dictionary, business rule, or "
        "domain confirmation) indicates these values are erroneous. Deleting "
        "them would be guessing intent not supported by the data.\n"
        "  3. Instead of deletion, distortion from extreme values will be "
        "neutralized statistically via log1p transform + standardization "
        "(Phase 4, Steps 9-10), which compresses scale without discarding "
        "any customer records.\n\n"
        "This choice is documented here explicitly per the implementation plan, "
        "for inclusion in the final report's Methodology section."
    )
    print(decision)
    return decision

## SECTION 4: Outlier Visualization Reference (Step 8, supports 8.5)

In [ ]:
# SECTION 4: Outlier Visualization Reference (Step 8, supports 8.5)
# ===========================================================================
def plot_outlier_reference(df: pd.DataFrame, outlier_summary: pd.DataFrame) -> None:
    """Boxplots annotated with outlier counts, for the report's outlier
    discussion (visual companion to Phase 2's boxplots)."""
    print("\n" + "=" * 70)
    print("SECTION 4: OUTLIER VISUALIZATION REFERENCE")
    print("=" * 70)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    for i, col in enumerate(SPEND_COLS):
        sns.boxplot(y=df[col], ax=axes[i], color="salmon")
        n_out = outlier_summary.loc[col, "n_outliers"]
        pct_out = outlier_summary.loc[col, "pct_outliers"]
        axes[i].set_title(f"{col}\n({n_out} outliers, {pct_out}%)")
        axes[i].set_ylabel(f"{col} (annual spend)")
    fig.suptitle("Outlier Reference: Boxplots Annotated with IQR Outlier Counts",
                 fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/06_outlier_annotated_boxplots.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/06_outlier_annotated_boxplots.png")

## SECTION 5: Multicollinearity Analysis — Correlation + VIF (Step 8.6)

In [ ]:
# SECTION 5: Multicollinearity Analysis — Correlation + VIF (Step 8.6)
# ===========================================================================
def compute_vif(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Compute Variance Inflation Factor per feature manually:
    VIF_i = 1 / (1 - R^2_i), where R^2_i is from regressing feature i on
    all other features in `cols`. Equivalent to statsmodels' VIF."""
    X = df[cols].values
    vifs = []
    for i, col in enumerate(cols):
        y = X[:, i]
        X_others = np.delete(X, i, axis=1)
        reg = LinearRegression().fit(X_others, y)
        r2 = reg.score(X_others, y)
        vif = np.inf if r2 == 1 else 1 / (1 - r2)
        vifs.append(vif)
    return pd.DataFrame({"feature": cols, "VIF": vifs}).set_index("feature")


def analyze_multicollinearity(df: pd.DataFrame) -> pd.DataFrame:
    """Quantify multicollinearity among the 6 spend features via correlation
    (already computed in Phase 2) and VIF (new, more rigorous diagnostic)."""
    print("\n" + "=" * 70)
    print("SECTION 5: MULTICOLLINEARITY ANALYSIS (CORRELATION + VIF)")
    print("=" * 70)

    corr = df[SPEND_COLS].corr()
    print("Correlation matrix (carried over from Phase 2):")
    print(corr.round(2).to_string())

    vif_df = compute_vif(df, SPEND_COLS)
    print("\nVariance Inflation Factor (VIF) per feature:")
    print(vif_df.round(2).to_string())
    print("\nInterpretation guide: VIF > 5 = moderate concern; VIF > 10 = high concern")
    for col, row in vif_df.iterrows():
        level = "HIGH" if row["VIF"] > 10 else ("MODERATE" if row["VIF"] > 5 else "LOW")
        print(f"  {col:<20} VIF = {row['VIF']:6.2f}  -> {level} multicollinearity")

    return vif_df

## SECTION 6: Multicollinearity Treatment Decision (Step 8.6)

In [ ]:
# SECTION 6: Multicollinearity Treatment Decision (Step 8.6)
# ===========================================================================
def document_multicollinearity_decision(vif_df: pd.DataFrame) -> str:
    """State and justify the multicollinearity-handling decision, comparing
    the two options laid out in the implementation plan."""
    print("\n" + "=" * 70)
    print("SECTION 6: MULTICOLLINEARITY TREATMENT DECISION")
    print("=" * 70)

    highest_vif_feature = vif_df["VIF"].idxmax()
    highest_vif_value = vif_df["VIF"].max()

    decision = (
        f"OBSERVED: Grocery, Milk, and Detergents_Paper show strong pairwise "
        f"correlation (see Phase 2). VIF, used here as a descriptive redundancy diagnostic rather than a K-Means requirement, also flags correlated features: "
        f"'{highest_vif_feature}' has the highest VIF at {highest_vif_value:.2f}.\n\n"
        "OPTIONS CONSIDERED:\n"
        "  Option A - PCA-assisted diagnostics: Keep all 6 features for clustering and profiling,\n"
        "             and use PCA separately for visualization/dimensionality diagnostics.\n"
        "             This does not remove correlation from the K-Means input; it preserves all\n"
        "             business-relevant spend categories while making low-dimensional structure visible.\n"
        "  Option B - Drop feature: Remove Detergents_Paper (most redundant with\n"
        "             Grocery, VIF/correlation-wise) and cluster on the remaining\n"
        "             5 features. Simpler, but permanently discards that spend\n"
        "             category, so it could no longer be reported per cluster.\n\n"
        "DECISION: Retain all six features and use PCA only as a visualization/diagnostic tool.\n"
        "RATIONALE:\n"
        "  1. No feature is dropped, so Phase 7's per-cluster profiling can still\n"
        "     report actual Detergents_Paper spend per cluster (business-relevant).\n"
        "  2. PCA directly supports Phase 8's 2D cluster visualization requirement\n"
        "     (Step 22-23) — one transformation serves two needs.\n"
        "  3. K-Means will still be run on the full standardized 6-feature space\n"
        "     (Phase 5-6) so cluster boundaries reflect all available spend\n"
        "     information, not just 2 compressed dimensions; PCA is used for\n"
        "     visualization/diagnostics, not as a replacement feature set.\n\n"
        "This decision, and the VIF table above, will be included in the final\n"
        "report's Methodology section to justify the choice over Option B."
    )
    print(decision)
    return decision

## MAIN — run Phase 3 end to end

In [ ]:
# MAIN — run Phase 3 end to end
# ===========================================================================
if __name__ == "__main__":
    skew_vals = analyze_skewness(df_raw)
    outlier_summary = detect_outliers_iqr(df_raw)
    outlier_decision = document_outlier_decision()
    plot_outlier_reference(df_raw, outlier_summary)
    vif_df = analyze_multicollinearity(df_raw)
    multicollinearity_decision = document_multicollinearity_decision(vif_df)

    print("\n" + "=" * 70)
    print("PHASE 3 COMPLETE")
    print("=" * 70)
    print("[OK] Skewness quantified — log1p transform justified for Phase 4.")
    print("[OK] Outliers quantified — decision: retain + neutralize via transform/scaling.")
    print("[OK] Multicollinearity quantified (correlation + VIF) — decision: PCA-assisted (Option A).")
    print("[OK] Ready for Phase 4 (Feature Transformation & Scaling).")

### Phase 3 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.